In [2]:
import hashlib
import json
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator

# Importaciones de LangChain (Asegúrate de tener instalado langchain-ollama)
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# =====================================================================
# 1. CONFIGURACIÓN DIRECTA EN EL NOTEBOOK
# =====================================================================
# Ajusta el modelo y la URL según tu entorno de Ollama o API
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3"  # o el modelo que estés usando localmente

# =====================================================================
# 2. ESQUEMAS ESTRICTOS DE PYDANTIC (Arquitectura de la Tesis)
# =====================================================================

class UbicacionSchema(BaseModel):
    ciudad: Optional[str] = Field(default=None, description="Ciudad o localidad específica")
    municipio: Optional[str] = Field(default=None, description="Municipio del Estado Bolívar")
    estado: Optional[str] = Field(default="Bolívar", description="Estado o provincia")
    pais: Optional[str] = Field(default="Venezuela", description="País")

class RecursoMultimediaSchema(BaseModel):
    enlace: str = Field(description="Nombre del archivo o URL (ej. autor.jpg, audio.mp3)")
    tipo: str = Field(description="Tipo de recurso: 'imagen', 'audio'")
    restriccion: str = Field(default="público", description="Acceso")

class ObraRelacionadaSchema(BaseModel):
    titulo: str = Field(description="Título de la obra literaria")
    genero: Optional[str] = Field(default=None, description="Género literario")
    fecha_publicacion: Optional[str] = Field(default=None, description="Año de publicación (YYYY)")
    lugar_publicacion: Optional[UbicacionSchema] = Field(default=None, description="Lugar de publicación")

class CriticaRelacionadaSchema(BaseModel):
    autor_critica: str = Field(description="Persona que escribe la crítica o reseña")
    titulo_critica: str = Field(description="Título del artículo o crítica")
    fuente: Optional[str] = Field(default=None, description="Medio de publicación")

class FichaLiterariaSchema(BaseModel):
    """Esquema Maestro del Autor - 16 Atributos Obligatorios [Tabla Ap-F1]"""
    nombres: str = Field(description="Nombres de pila del autor (PROHIBIDO crear 'nombre_completo')")
    apellidos: str = Field(description="Apellidos del autor")
    sexo: Optional[str] = Field(default=None, description="Sexo o género del autor")
    seudonimo: Optional[str] = Field(default=None, description="Seudónimo literario")
    
    fechaDeNacimiento: Optional[str] = Field(default=None, description="Fecha de nacimiento (YYYY-MM-DD)")
    lugarDeNacimiento: Optional[UbicacionSchema] = Field(default=None, description="Lugar de nacimiento desglosado")
    fechaDeFallecimiento: Optional[str] = Field(default=None, description="Fecha de muerte (None si está vivo)")
    lugarDeFallecimiento: Optional[UbicacionSchema] = Field(default=None, description="Lugar de fallecimiento desglosado")
    
    actividadRelevante: List[str] = Field(default_factory=list, description="Estudios, cargos o profesiones")
    familiaresDestacados: List[str] = Field(default_factory=list, description="Familiares mencionados")
    contextoEnQueVivio: Optional[str] = Field(default=None, description="Entorno histórico, social o político")
    tematicaPrincipal: List[str] = Field(default_factory=list, description="Temas recurrentes en su obra")
    genero_principal: Optional[str] = Field(default=None, description="Máximo exponente de: Poesía, Narrativa, etc.")
    
    resumen_biografico: str = Field(description="Redacción académica, fluida y coherente generada por el LLM.")
    
    imagenAutor: Optional[RecursoMultimediaSchema] = Field(default=None, description="Referencia a la foto (.jpg)")
    audioVoz: Optional[RecursoMultimediaSchema] = Field(default=None, description="Referencia al archivo de voz (.mp3)")
    
    obras: List[ObraRelacionadaSchema] = Field(default_factory=list, description="Obras detectadas (:TIENE_OBRAS)")
    criticas: List[CriticaRelacionadaSchema] = Field(default_factory=list, description="Críticas detectadas (:CRITICA_SOBRE)")

    @field_validator("sexo", "seudonimo", "genero_principal", "contextoEnQueVivio")
    @classmethod
    def limpiar_valores_vacios(cls, v):
        if v in ["", "None", "null"]:
            return None
        return v

# =====================================================================
# 3. LÓGICA DE IDENTIFICADORES Y PIPELINE
# =====================================================================

def generar_ficha_id(nombres: str, apellidos: str) -> str:
    limpio = f"{nombres.strip()}_{apellidos.strip()}".upper().replace(" ", "")
    hash_txt = hashlib.md5(limpio.encode('utf-8')).hexdigest()[:8].upper()
    return f"AUTOR_{limpio}_{hash_txt}"

def extraer_json_de_ficha(texto_ficha: str) -> dict:
    # Inicialización del modelo estructurado
    llm = ChatOllama(base_url=OLLAMA_BASE_URL, model=OLLAMA_MODEL, temperature=0)
    llm_estructurado = llm.with_structured_output(FichaLiterariaSchema)

    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "Eres el Agente Extractor Core del proyecto GraphRAG Literario del Estado Bolívar.\n"
            "Mapea el texto directamente al esquema estricto de la tesis.\n\n"
            "REGLAS CRÍTICAS:\n"
            "1. NO agregues bajo ninguna circunstancia el campo 'nombre_completo'.\n"
            "2. Desglosa geográficamente las ubicaciones (ciudad, municipio, estado, pais).\n"
            "3. Genera referencias multimedia (.jpg, .mp3) si el texto indica que existen.\n"
            "4. Redacta un 'resumen_biografico' coherente y fluido, no una lista de datos.\n"
            "5. Extrae la lista de obras y críticas en sus respectivas estructuras si existen."
        )),
        ("human", "{texto}")
    ])

    cadena = prompt | llm_estructurado
    datos_extraidos = cadena.invoke({"texto": texto_ficha})
    
    # Inyectar metadatos dinámicos
    resultado = datos_extraidos.model_dump()
    resultado["fichaId"] = generar_ficha_id(resultado["nombres"], resultado["apellidos"])
    
    return resultado


In [3]:
# =====================================================================
# 4. PRUEBA DE EJECUCIÓN (Ejemplo de Ficha Literaria para el Bolívar)
# =====================================================================

texto_ejemplo_ficha = """
Ficha Biográfica: César Augusto Beltrán Vinueza. Nacido en Upata (Municipio Piar, Estado Bolívar) 
el 12 de marzo de 1954. Destacado poeta y cronista de la región guayanesa, cuya obra se centra 
en la majestuosidad de la selva y las tradiciones de los pueblos mineros del sur. Se conserva 
un retrato en óleo digitalizado del autor (imagen_cesar_beltran.jpg). 
Entre sus publicaciones más célebres destaca el poemario 'Ecos de la Piedra' publicado en Ciudad Bolívar en 1982. 
El crítico literario Juan Ortiz escribió una reseña laudatoria sobre este poemario titulada 'La Voz del Sur' 
en la Revista de Literatura Regional.
"""

# Ejecutar la extracción en el cuaderno
print("⏳ Procesando extracción estructurada con el LLM...")
resultado_json = extraer_json_de_ficha(texto_ejemplo_ficha)

# Imprimir el resultado con sangría limpia para inspección visual directa
print("\n✅ Extracción Exitosa. JSON Generado:")
print(json.dumps(resultado_json, indent=4, ensure_ascii=False))

⏳ Procesando extracción estructurada con el LLM...


ConnectError: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión